### Лабораторна робота №5: Абелеві групи та кільця

**Мета роботи:** Практично вивчити властивості абелевих груп, навчитися працювати з кільцями та визначати їх структуру за допомогою SageMath.

---
### 1. Структура абелевих груп

Розглянемо абелеві групи порядку $400$:
$$
\mathbb{Z}_{400}, \quad \mathbb{Z}_{16}\times\mathbb{Z}_{25}, \quad \mathbb{Z}_{20}\times\mathbb{Z}_{20}, \quad \mathbb{Z}_{4}\times\mathbb{Z}_{100}, \quad \mathbb{Z}_{2}\times\mathbb{Z}_{10}\times\mathbb{Z}_{20}, \quad \mathbb{Z}_{505}^{*}.
$$
Для кожної з цих груп виконайте наступні кроки:
1. Знайдіть елементарні дільники групи.
2. Розкладіть групу у добуток циклічних $p$-груп.
3. Визначте кількість елементів кожного можливого порядку.
4. Порівняйте групи та визначте, які з них є ізоморфними.


In [1]:
from collections import Counter

# Z_505^*: порядки циклічних множників беремо з unit_group()
orders_505 = list(Integers(505).unit_group().gens_orders())
print("|Z_505^*| =", Integers(505).unit_group_order())   # перевірка: має бути 400

groups = {
    "Z_400":             [400],
    "Z_16 x Z_25":       [16, 25],
    "Z_20 x Z_20":       [20, 20],
    "Z_4 x Z_100":       [4, 100],
    "Z_2 x Z_10 x Z_20": [2, 10, 20],
    "Z_505^*":           orders_505,
}

def elementary_divisors(orders):
    """Елементарні дільники: степені простих p^e з розкладу кожного n_i."""
    return sorted(p^e for n in orders for p, e in factor(n))

classes = {}   # елементарні дільники -> список груп з такими дільниками
for name, orders in groups.items():
    G = AbelianGroup(orders)
    ed = elementary_divisors(orders)
    print(name, "| порядок =", G.order())
    print("  елементарні дільники:", ed)
    print("  розклад у p-групи:  ", " x ".join(f"Z_{d}" for d in ed))
    cnt = Counter(g.order() for g in G)
    print("  (порядок, кількість елементів):", sorted(cnt.items()))
    classes.setdefault(tuple(ed), []).append(name)

# Дві скінченні абелеві групи ізоморфні <=> збігаються їхні елементарні дільники
print("\nКласи ізоморфізму:")
for ed, names in classes.items():
    print("  ", " ≅ ".join(names), "  ->", list(ed))


|Z_505^*| = 400
Z_400 | порядок = 400
  елементарні дільники: [16, 25]
  розклад у p-групи:   Z_16 x Z_25
  (порядок, кількість елементів): [(1, 1), (2, 1), (4, 2), (5, 4), (8, 4), (10, 4), (16, 8), (20, 8), (25, 20), (40, 16), (50, 20), (80, 32), (100, 40), (200, 80), (400, 160)]
Z_16 x Z_25 | порядок = 400
  елементарні дільники: [16, 25]
  розклад у p-групи:   Z_16 x Z_25
  (порядок, кількість елементів): [(1, 1), (2, 1), (4, 2), (5, 4), (8, 4), (10, 4), (16, 8), (20, 8), (25, 20), (40, 16), (50, 20), (80, 32), (100, 40), (200, 80), (400, 160)]
Z_20 x Z_20 | порядок = 400
  елементарні дільники: [4, 4, 5, 5]
  розклад у p-групи:   Z_4 x Z_4 x Z_5 x Z_5
  (порядок, кількість елементів): [(1, 1), (2, 3), (4, 12), (5, 24), (10, 72), (20, 288)]
Z_4 x Z_100 | порядок = 400
  елементарні дільники: [4, 4, 25]
  розклад у p-групи:   Z_4 x Z_4 x Z_25
  (порядок, кількість елементів): [(1, 1), (2, 3), (4, 12), (5, 4), (10, 12), (20, 48), (25, 20), (50, 60), (100, 240)]
Z_2 x Z_10 x Z_20 | пор

---
### 2. Одна абелева група порядку $n$
1. Реалізуйте функцію, яка для заданого натурального числа $n$ обчислює кількість попарно неізоморфних абелевих груп порядку $n$.
2. Знайдіть перші сто таких натуральних чисел $n$, для яких існує рівно одна абелева група порядку $n$. Для кожного знайденого $n$ проаналізуйте його розклад на прості множники.
3. Сформулюйте гіпотезу: для яких $n\in\mathbb{N}$ існує рівно одна (з точністю до ізоморфізму) абелева група порядку $n$.


In [ ]:
def count_abelian_groups(n):
    """Кількість абелевих груп порядку n: добуток p(e) по розкладу n = prod p^e,
    де p(e) - кількість розбиттів числа e."""
    return prod(number_of_partitions(e) for p, e in factor(n))

unique = []
n = 1
while len(unique) < 100:
    if count_abelian_groups(n) == 1:
        unique.append(n)
    n += 1

for n in unique:
    print(n, "=", factor(n))

print("\nУсі знайдені n вільні від квадратів:", all(n.is_squarefree() for n in unique))
print("Гіпотеза вірна для всіх n <= 10000:",
      all((count_abelian_groups(n) == 1) == n.is_squarefree() for n in srange(1, 10001)))


---
### 3. Дільники нуля та одиниці в скінченних кільцях
Розгляньте скінченні кільця $\mathbb{Z}_{30}$, $\mathbb{Z}_{12}\times\mathbb{Z}_{15}$, $M_3(\mathbb{Z}_3)$ та для кожного з них виконайте наступні кроки:
1. Знайдіть дільники нуля та їх кількість.
2. Знайдіть дільники одиниці та їх кількість.
3. Порівняйте отримані результати та зробіть висновок про взаємозв'язок між дільниками нуля та дільниками одиниці в скінченних кільцях.


In [ ]:
def zero_divisors(elems, mul, zero):
    return [a for a in elems if a != zero and any(mul(a, b) == zero for b in elems if b != zero)]

def units(elems, mul, one):
    return [a for a in elems if any(mul(a, b) == one for b in elems)]

def report(name, total, zd, un):
    print(name)
    print("  дільників нуля:", len(zd), "| дільників одиниці:", len(un))
    print("  1 + #дільників нуля + #дільників одиниці =", 1 + len(zd) + len(un), "| |R| =", total)

R = Integers(30)
E = list(R)
zd = zero_divisors(E, lambda a, b: a*b, R(0))
un = R.list_of_elements_of_multiplicative_group()  
report("Z_30", len(E), zd, un)
print("  дільники нуля:", zd)
print("  дільники одиниці:", un)

R1, R2 = Integers(12), Integers(15)
E = [(a, b) for a in R1 for b in R2]
mul = lambda x, y: (x[0]*y[0], x[1]*y[1])
zd = zero_divisors(E, mul, (R1(0), R2(0)))
un = units(E, mul, (R1(1), R2(1)))
report("Z_12 x Z_15", len(E), zd, un)


M = MatrixSpace(GF(3), 3, 3)
E = list(M)
zd = [A for A in E if A != 0 and A.det() == 0]
un = [A for A in E if A.det() != 0]
report("M_3(Z_3)", len(E), zd, un)

print("\nУ скінченному кільці з 1 кожен ненульовий елемент є або дільником нуля, "
      "або дільником одиниці (але не обома одразу): R = {0} ∪ {дільники нуля} ∪ {дільники одиниці}.")


---
### 4. Факторкільця кільця многочленів
Нехай $f(x)=x^4+x+2$. Виконайте наступні кроки для кожного з кілець $R=\mathbb{Q}, \mathbb{Z}_4, \mathbb{Z}_5$:
1. Побудуйте ідеал $I=(f(x))$ у кільці $R[x]$.
2. Побудуйте факторкільце $R/I$ та знайдіть його порядок.
3. Визначте, чи є факторкільце $R/I$ полем.
4. Знайдіть дільники нуля у факторкільці $R/I$.
5. Розкладіь $f(x)$ на лінійні множники над $R$ та порівняйте з дільниками нуля у факторкільці $R/I$.
6. Перевірте, чи елемент $x+I\in R/I$ є оборотним.

In [ ]:
from itertools import product

f_str = "x^4 + x + 2"

def analyze(R):
    print("=" * 60)
    print("R =", R)
    P = PolynomialRing(R, 'x')
    x = P.gen()
    f = x^4 + x + 2

    I = P.ideal(f)
    print("I =", I)

    Q = P.quotient(I, 'a')
    print("R[x]/I =", Q)
    print("порядок:", R.cardinality()^f.degree() if R.is_finite() else "нескінченне")

    is_field = R.is_field() and f.is_irreducible()
    print("є полем:", is_field)

    if R.is_field():
        print("розклад f:", f.factor())
        roots = f.roots(multiplicities=False)
    else:
        roots = [r for r in R if f(r) == 0]
    print("корені f у R:", roots, "-> лінійні множники:", [x - r for r in roots])

    if R.is_finite():
        nonzero = [P(list(c)) for c in product(R, repeat=4) if any(c)]   
        zd = [u for u in nonzero if any((u*v) % f == 0 for v in nonzero)]
        print("дільників нуля:", len(zd), "| приклади:", zd[:5])
        for r in roots:
            print(f"  x - {r} є дільником нуля в R/I:", (x - r) in zd)
        inv_x = any((x*v) % f == 1 for v in nonzero)
    else:
        print("дільників нуля немає (R/I - поле)")
        inv_x = x.gcd(f) == 1

    print("x + I оборотний:", inv_x)

for R in [QQ, Integers(4), Integers(5)]:
    analyze(R)


---
### 5*. Корінь з $-1$ за модулем $p$

Виконайте наступні кроки і на їх основі сформулюйте гіпотезу: для яких простих чисел $p$ фактор-кільце $\mathbb{Z}_p[x]/(x^2+1)$ є полем, тобто многочлен $x^2+1$ не має коренів у $\mathbb{Z}_p$.
1. Знайдіть перші 100 простих чисел $p$, для яких факторкільце $\mathbb{Z}_p[x]/(x^2+1)$ є полем.
2. Визначте остачі цих простих чисел при діленні на $m=2,3,\ldots,7$ та проаналізуйте отримані результати.


---
### 6*. Розфарбування кубика Рубика $2\times2\times2$

Розглянемо кубик Рубика розміру $2\times2\times2$, який має $24$ зовнішні грані кутиків (наліпки):

![rubik cube](../assets/rubik_cube.jpg)

1. Побудуйте групу поворотів кубика Рубика 2х2х2 як підгрупу симетричної групи $S_{24}$ та знайдіть її порядок.
2. Знайдіть кількість різних розфарбувань кубика у 10 кольорів. Вважайте два розфарбування однаковими, якщо одне можна отримати з іншого за допомогою поворотів кубика.



In [ ]:
G = SymmetricGroup(24)
g1 = G('(1, 2, 4, 3)(5, 24, 9, 7)(6, 23, 10, 8)');
...
H = G.subgroup([g1, g2, g3, g4, g5, g6]);
H.order()